<a href="https://colab.research.google.com/github/KhushbooS03/KhushbooS03/blob/main/batch/AlphaFold2_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#ColabFold v1.6.3: AlphaFold2 w/ MMseqs2 BATCH

<img src="https://raw.githubusercontent.com/sokrypton/ColabFold/main/.github/ColabFold_Marv_Logo_Small.png" height="256" align="right" style="height:256px">

Easy to use AlphaFold2 protein structure [(Jumper et al. 2021)](https://www.nature.com/articles/s41586-021-03819-2) and complex [(Evans et al. 2021)](https://www.biorxiv.org/content/10.1101/2021.10.04.463034v1) prediction using multiple sequence alignments generated through MMseqs2. For details, refer to our manuscript:

[Mirdita M, Schütze K, Moriwaki Y, Heo L, Ovchinnikov S, Steinegger M. ColabFold: Making protein folding accessible to all.
*Nature Methods*, 2022](https://www.nature.com/articles/s41592-022-01488-1)

**Usage**

`input_dir` directory with only fasta files or MSAs stored in Google Drive. MSAs need to be A3M formatted and have an `.a3m` extention. For MSAs MMseqs2 will not be called.

`result_dir` results will be written to the result directory in Google Drive

Old versions: [v1.4](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.4.0/batch/AlphaFold2_batch.ipynb), [v1.5.1](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.1/batch/AlphaFold2_batch.ipynb), [v1.5.2](https://colab.research.google.com/github/sokrypton/ColabFold/blob/v1.5.2/batch/AlphaFold2_batch.ipynb), [v1.5.3-patch](https://colab.research.google.com/github/sokrypton/ColabFold/blob/56c72044c7d51a311ca99b953a71e552fdc042e1/batch/AlphaFold2_batch.ipynb)

<strong>For more details, see <a href="#Instructions">bottom</a> of the notebook and checkout the [ColabFold GitHub](https://github.com/sokrypton/ColabFold). </strong>

-----------

### News
- <b><font color='green'>2023/07/31: The ColabFold MSA server is back to normal. It was using older DB (UniRef30 2202/PDB70 220313) from 27th ~8:30 AM CEST to 31st ~11:10 AM CEST.</font></b>
- <b><font color='green'>2023/06/12: New databases! UniRef30 updated to 2023_02 and PDB to 230517. We now use PDB100 instead of PDB70 (see notes in the [main](https://colabfold.com) notebook).</font></b>
- <b><font color='green'>2023/06/12: We introduced a new default pairing strategy: Previously, for multimer predictions with more than 2 chains, we only pair if all sequences taxonomically match ("complete" pairing). The new default "greedy" strategy pairs any taxonomically matching subsets.</font></b>

In [1]:
#@title Mount google drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#@title Input protein sequence, then hit `Runtime` -> `Run all`

input_dir = '/content/drive/MyDrive/Alphafold/Alphafold_batch1.faa' #@param {type:"string"}
result_dir = '/content/drive/MyDrive/results_batch1' #@param {type:"string"}

# number of models to use
#@markdown ---
#@markdown ### Advanced settings
msa_mode = "MMseqs2 (UniRef+Environmental)" #@param ["MMseqs2 (UniRef+Environmental)", "MMseqs2 (UniRef only)","single_sequence","custom"]
num_models = 5 #@param [1,2,3,4,5] {type:"raw"}
num_recycles = 3 #@param [1,3,6,12,24,48] {type:"raw"}
stop_at_score = 100 #@param {type:"string"}
#@markdown - early stop computing models once score > threshold (avg. plddt for "structures" and ptmscore for "complexes")
use_custom_msa = False
num_relax = 1 #@param [0, 1, 5] {type:"raw"}
use_amber = num_relax > 0
relax_max_iterations = 200 #@param [0,200,2000] {type:"raw"}
use_templates = False #@param {type:"boolean"}
do_not_overwrite_results = True #@param {type:"boolean"}
zip_results = False #@param {type:"boolean"}
use_fast_kernels = False #@param {type:"boolean"}
#@markdown - use fused kernels for faster prediction (experimental)


In [3]:
#@title Install dependencies
%%bash -s $use_amber

set -e

USE_AMBER=$1

EXTRAS="alphafold-minus-jax"
OPENMM=""
READY=COLABFOLD_READY
if [ ${USE_AMBER} == "True" ]; then
  EXTRAS="alphafold-minus-jax,openmm"
  OPENMM=$(python -c "from importlib.metadata import distributions; i={d.metadata['Name'] for d in distributions()}; print(next((f'openmm[{c}]' for c in ('cuda13','cuda12') if f'jax-{c}-plugin' in i), ''))")
  READY=COLABFOLD_AMBER_READY
fi

if [ ! -f ${READY} ]; then
  # install dependencies
  # We have to use "--no-warn-conflicts" because colab already has a lot preinstalled with requirements different to ours
  pip install -q --no-warn-conflicts "colabfold[${EXTRAS}] @ git+https://github.com/sokrypton/ColabFold" ${OPENMM}
  if [ -n "${TPU_NAME}" ]; then
    pip install -q --no-warn-conflicts -U dm-haiku==0.0.10 jax==0.3.25
  fi
  ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold
  ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold
  touch ${READY}
fi

# Download params (~1min)
python -m colabfold.download


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 665.9/665.9 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.0/270.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 117.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 377.0/377.0 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.4/14.4 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.3/131.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 11.6 MB/s eta 0:00:00


In [4]:
#@title Run Prediction

import sys

from colabfold.batch import get_queries, run
from colabfold.download import default_data_dir
from colabfold.utils import setup_logging
from pathlib import Path

setup_logging(Path(result_dir).joinpath("log.txt"))

queries, is_complex = get_queries(input_dir)
run(
    queries=queries,
    result_dir=result_dir,
    use_templates=use_templates,
    num_relax=num_relax,
    relax_max_iterations=relax_max_iterations,
    msa_mode=msa_mode,
    model_type="auto",
    num_models=num_models,
    num_recycles=num_recycles,
    model_order=[1, 2, 3, 4, 5],
    is_complex=is_complex,
    data_dir=default_data_dir,
    keep_existing_results=do_not_overwrite_results,
    rank_by="auto",
    pair_mode="unpaired+paired",
    stop_at_score=stop_at_score,
    zip_results=zip_results,
    use_fast_kernels=use_fast_kernels,
    user_agent="colabfold/google-colab-batch",
)

2026-10-06 09:38:17,253 Running on GPU
2026-10-06 09:38:18,359 Found 6 citations for tools or databases
2026-10-06 09:38:18,361 Query 1/3: MHF1860796.1_eggNOG_NHADNH_05058_status_reliable_same_strand (length 256)


COMPLETE: 100%|██████████| 150/150 [elapsed: 00:04 remaining: 00:00]


2026-10-06 09:38:39,066 Padding length to 266
2026-10-06 09:39:29,510 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=92.2 pTM=0.875
2026-10-06 09:39:51,472 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=93.2 pTM=0.88 tol=0.629
2026-10-06 09:40:13,924 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=93.2 pTM=0.882 tol=0.741
2026-10-06 09:40:36,659 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=93.2 pTM=0.882 tol=0.714
2026-10-06 09:40:36,660 alphafold2_ptm_model_1_seed_000 took 117.6s (3 recycles)
2026-10-06 09:40:59,969 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=92.5 pTM=0.879
2026-10-06 09:41:23,507 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=93.3 pTM=0.893 tol=1.57
2026-10-06 09:41:47,339 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=93.5 pTM=0.897 tol=0.68
2026-10-06 09:42:11,477 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=93.6 pTM=0.899 tol=1.04
2026-10-06 09:42:11,478 alphafold2_ptm_model_2_seed_000 took 94.8s (3 recycles)
2026-10-06 09:42:35,975 alphafold2_ptm_mo

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:03 remaining: 00:00]


2026-10-06 09:47:43,588 Padding length to 326
2026-10-06 09:48:51,785 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=89.5 pTM=0.871
2026-10-06 09:49:24,231 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=90.6 pTM=0.88 tol=1.01
2026-10-06 09:49:57,308 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=90.7 pTM=0.883 tol=0.693
2026-10-06 09:50:30,151 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=91.3 pTM=0.886 tol=0.414
2026-10-06 09:50:30,152 alphafold2_ptm_model_1_seed_000 took 166.6s (3 recycles)
2026-10-06 09:51:02,924 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=89.6 pTM=0.876
2026-10-06 09:51:35,835 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=91 pTM=0.888 tol=1.01
2026-10-06 09:52:08,594 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=90.7 pTM=0.887 tol=0.425
2026-10-06 09:52:41,386 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=91.1 pTM=0.889 tol=0.179
2026-10-06 09:52:41,387 alphafold2_ptm_model_2_seed_000 took 131.2s (3 recycles)
2026-10-06 09:53:14,342 alphafold2_ptm_mo

COMPLETE: 100%|██████████| 150/150 [elapsed: 00:01 remaining: 00:00]


2026-10-06 10:00:29,985 alphafold2_ptm_model_1_seed_000 recycle=0 pLDDT=88.6 pTM=0.886
2026-10-06 10:01:02,426 alphafold2_ptm_model_1_seed_000 recycle=1 pLDDT=88.2 pTM=0.885 tol=0.93
2026-10-06 10:01:35,571 alphafold2_ptm_model_1_seed_000 recycle=2 pLDDT=88.4 pTM=0.887 tol=0.761
2026-10-06 10:02:08,457 alphafold2_ptm_model_1_seed_000 recycle=3 pLDDT=88.6 pTM=0.887 tol=0.297
2026-10-06 10:02:08,458 alphafold2_ptm_model_1_seed_000 took 130.5s (3 recycles)
2026-10-06 10:02:41,396 alphafold2_ptm_model_2_seed_000 recycle=0 pLDDT=91.5 pTM=0.906
2026-10-06 10:03:14,300 alphafold2_ptm_model_2_seed_000 recycle=1 pLDDT=91.6 pTM=0.909 tol=0.685
2026-10-06 10:03:47,289 alphafold2_ptm_model_2_seed_000 recycle=2 pLDDT=91.3 pTM=0.909 tol=0.466
2026-10-06 10:04:20,229 alphafold2_ptm_model_2_seed_000 recycle=3 pLDDT=90.9 pTM=0.905 tol=0.174
2026-10-06 10:04:20,230 alphafold2_ptm_model_2_seed_000 took 131.7s (3 recycles)
2026-10-06 10:04:53,259 alphafold2_ptm_model_3_seed_000 recycle=0 pLDDT=92.6 pTM=0.

{'rank': [['rank_001_alphafold2_ptm_model_2_seed_000',
   'rank_002_alphafold2_ptm_model_3_seed_000',
   'rank_003_alphafold2_ptm_model_1_seed_000',
   'rank_004_alphafold2_ptm_model_4_seed_000',
   'rank_005_alphafold2_ptm_model_5_seed_000'],
  ['rank_001_alphafold2_ptm_model_5_seed_000',
   'rank_002_alphafold2_ptm_model_4_seed_000',
   'rank_003_alphafold2_ptm_model_3_seed_000',
   'rank_004_alphafold2_ptm_model_1_seed_000',
   'rank_005_alphafold2_ptm_model_2_seed_000'],
  ['rank_001_alphafold2_ptm_model_3_seed_000',
   'rank_002_alphafold2_ptm_model_4_seed_000',
   'rank_003_alphafold2_ptm_model_2_seed_000',
   'rank_004_alphafold2_ptm_model_5_seed_000',
   'rank_005_alphafold2_ptm_model_1_seed_000']],
 'metric': [[{'mean_plddt': 93.5625,
    'ptm': 0.89892578125,
    'print_line': ' pLDDT=93.6 pTM=0.899'},
   {'mean_plddt': 93.3125,
    'ptm': 0.8876953125,
    'print_line': ' pLDDT=93.3 pTM=0.888'},
   {'mean_plddt': 93.25,
    'ptm': 0.8818359375,
    'print_line': ' pLDDT=93.2

# Instructions <a name="Instructions"></a>
**Quick start**
1. Upload your single fasta files to a folder in your Google Drive
2. Define path to the fold containing the fasta files (`input_dir`) define an outdir (`output_dir`)
3. Press "Runtime" -> "Run all".

**Result zip file contents**

At the end of the job a all results `jobname.result.zip` will be uploaded to your (`output_dir`) Google Drive. Each zip contains one protein.

1. PDB formatted structures sorted by avg. pIDDT. (unrelaxed and relaxed if `use_amber` is enabled).
2. Plots of the model quality.
3. Plots of the MSA coverage.
4. Parameter log file.
5. A3M formatted input MSA.
6. BibTeX file with citations for all used tools and databases.


**Troubleshooting**
* Check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check your input sequence.

**Known issues**
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Google Colab assigns different types of GPUs with varying amount of memory. Some might not have enough memory to predict the structure for a long sequence.
* Your browser can block the pop-up for downloading the result file. You can choose the `save_to_google_drive` option to upload to Google Drive instead or manually download the result file: Click on the little folder icon to the left, navigate to file: `jobname.result.zip`, right-click and select \"Download\" (see [screenshot](https://pbs.twimg.com/media/E6wRW2lWUAEOuoe?format=jpg&name=small)).

**Limitations**
* Computing resources: Our MMseqs2 API can handle ~20-50k requests per day.
* MSAs: MMseqs2 is very precise and sensitive but might find less hits compared to HHblits/HMMer searched against BFD or Mgnify.
* We recommend to additionally use the full [AlphaFold2 pipeline](https://github.com/deepmind/alphafold).

**Description of the plots**
*   **Number of sequences per position** - We want to see at least 30 sequences per position, for best performance, ideally 100 sequences.
*   **Predicted lDDT per position** - model confidence (out of 100) at each position. The higher the better.
*   **Predicted Alignment Error** - For homooligomers, this could be a useful metric to assess how confident the model is about the interface. The lower the better.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/sokrypton/ColabFold/issues

**License**

The source code of ColabFold is licensed under [MIT](https://raw.githubusercontent.com/sokrypton/ColabFold/main/LICENSE). Additionally, this notebook uses AlphaFold2 source code and its parameters licensed under [Apache 2.0](https://raw.githubusercontent.com/deepmind/alphafold/main/LICENSE) and  [CC BY 4.0](https://creativecommons.org/licenses/by-sa/4.0/) respectively. Read more about the AlphaFold license [here](https://github.com/deepmind/alphafold).

**Acknowledgments**
- We thank the AlphaFold team for developing an excellent model and open sourcing the software.

- Do-Yoon Kim for creating the ColabFold logo.

- A colab by Sergey Ovchinnikov ([@sokrypton](https://twitter.com/sokrypton)), Milot Mirdita ([@milot_mirdita](https://twitter.com/milot_mirdita)) and Martin Steinegger ([@thesteinegger](https://twitter.com/thesteinegger)).
